# EV Charging Anomaly Detection - Data Exploration

Let's look at the data and see if we can find anomalies.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

sys.path.insert(0, os.path.join('..', 'src'))
from features import load_data, engineer_features, get_feature_columns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 20)

## 1. Load the Data

Let's start by loading and looking at the data.

In [ ]:
df = load_data('../charging_logs.csv')
print(f"{len(df):,} events | {df['station_id'].nunique()} stations | {df['session_id'].nunique()} sessions")
print(f"Date range: {df['timestamp'].min()} → {df['timestamp'].max()}")
df.head()

In [ ]:
# Get basic stats about all columns
df.describe()

## 2. Look for Obvious Problems

What violations do we see in the data?

In [ ]:
print("Obvious problems we can spot:")
print()
print("Negative power_kw:", (df['power_kw'] < 0).sum())
print("  → Energy flowing backwards? That's wrong during charging")
print()
print("Zero current with non-zero power:", ((df['current'] == 0) & (df['power_kw'] > 0)).sum())
print("  → These readings don't match up")
print()
print("Voltage too low (< 190V):", (df['voltage'] < 190).sum())
print("  → Normal charging is around 208-240V")
print()
print("Voltage too high (> 260V):", (df['voltage'] > 260).sum())
print("  → Can damage equipment")
print()
print("Temperature over 75°C:", (df['temperature_c'] > 75).sum())
print("  → Getting too hot - thermal concern")
print()
print("Error codes present:", (df['error_code'] != 0).sum())
print("  → Charger reported a problem")
print()
print("Error code breakdown:")
print(df['error_code'].value_counts())

In [ ]:
# Plot distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Sensor Readings', fontsize=14)

for ax, col in zip(axes.flat, ['voltage', 'current', 'power_kw', 'temperature_c', 'energy_kwh', 'duration_sec']):
    ax.hist(df[col], bins=80, color='steelblue', edgecolor='none', alpha=0.8)
    ax.set_title(col)

plt.tight_layout()
plt.show()

print("Notice the tails in the distributions")
print("- Voltage drops below 190V")
print("- Power goes negative")
print("- Temperature spikes above 75°C")

## 3. Different Stations Behave Differently

Each station operates at different ranges. Need to account for this.

In [ ]:
# Look at station-level statistics
station_summary = df.groupby('station_id').agg(
    mean_voltage=('voltage', 'mean'),
    mean_temp=('temperature_c', 'mean'),
    mean_power=('power_kw', 'mean'),
    error_rate=('error_code', lambda x: (x != 0).mean()),
).round(2)
station_summary.sort_values('mean_temp', ascending=False)

print("\nKey insight:")
print("- Some stations run warmer than others")
print("- A temperature of 55°C is normal at one station but weird at another")
print("- This is why we use station-specific baselines in our features")

## 4. Build Features

Now we create features from the raw data. These help the model understand what's normal and what's not.

In [ ]:
df_feat = engineer_features(df)
print(f"Original columns: {len(df.columns)}")
print(f"New columns added: {len(df_feat.columns) - len(df.columns)}")
print(f"Total columns now: {len(df_feat.columns)}")
print()
print("Basic stats on the engineered features:")
df_feat[get_feature_columns()].describe()

In [ ]:
# Power factor should be around 0.95-1.0 for healthy chargers
pf = df_feat['power_factor'].dropna()
plt.figure(figsize=(8, 4))
plt.hist(pf.clip(-0.5, 2.0), bins=100, color='steelblue', edgecolor='none')
plt.axvline(0.95, color='red', linestyle='--', label='expected range start')
plt.axvline(1.0, color='green', linestyle='--', label='ideal')
plt.title('Power Factor Distribution')
plt.xlabel('power_factor')
plt.ylabel('count')
plt.legend()
plt.show()

print("Power factor should be around 0.95-1.0")
print("Values way off from this are probably bad")

## 5. Train the Model

Train Isolation Forest and see how many anomalies it finds.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Get the features we want to use
feature_cols = get_feature_columns()
X = df_feat[feature_cols].fillna(df_feat[feature_cols].median())

# Scale the features (required for the model)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train Isolation Forest
model = IsolationForest(n_estimators=200, contamination=0.025, random_state=42, n_jobs=-1)
model.fit(X_scaled)

# Get predictions
df_feat['if_anomaly'] = (model.predict(X_scaled) == -1).astype(int)

# Also apply hard rules
df_feat['hard_rule'] = (
    (df_feat['negative_power'] == 1) |
    (df_feat['current_zero'] == 1) |
    (df_feat['voltage_out_of_range'] == 1) |
    (df_feat['overtemp'] == 1) |
    (df_feat['has_error'] == 1)
).astype(int)

# Combine both
df_feat['is_anomaly'] = ((df_feat['if_anomaly'] == 1) | (df_feat['hard_rule'] == 1)).astype(int)

print("Results:")
print(f"Total flagged: {df_feat['is_anomaly'].sum():,} out of {len(df_feat):,}")
print(f"Percentage: {df_feat['is_anomaly'].mean()*100:.2f}%")
print()
print("Breakdown:")
print(f"  Hard rules only: {((df_feat['hard_rule']==1) & (df_feat['if_anomaly']==0)).sum():,}")
print(f"  Model only: {((df_feat['hard_rule']==0) & (df_feat['if_anomaly']==1)).sum():,}")
print(f"  Both: {((df_feat['hard_rule']==1) & (df_feat['if_anomaly']==1)).sum():,}")

In [ ]:
# Show anomaly rate by station
anomaly_by_station = df_feat.groupby('station_id')['is_anomaly'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 4))
anomaly_by_station.plot(kind='bar', color='steelblue')
plt.title('Anomaly Rate by Station')
plt.ylabel('percentage of events flagged')
plt.xlabel('station')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Different stations have different anomaly rates")
print("This makes sense - some stations have more problems than others")

In [ ]:
# Look at some actual flagged events
anomalies = df_feat[df_feat['is_anomaly'] == 1]

print("Sample of flagged anomalies:")
print()
sample = anomalies[['station_id','timestamp','voltage','current','power_kw','temperature_c','error_code','message']].sample(10, random_state=1)
display(sample)

In [ ]:
# Are there temporal patterns? Do anomalies happen more at certain times?
hourly = df_feat.groupby('hour')['is_anomaly'].mean()

plt.figure(figsize=(10, 4))
hourly.plot(kind='bar', color='steelblue')
plt.title('Anomaly Rate by Hour of Day')
plt.xlabel('hour (0=midnight, 12=noon)')
plt.ylabel('percentage flagged')
plt.tight_layout()
plt.show()

print("Do we see patterns in what time of day anomalies happen?")

## 6. Examples of What Gets Flagged

Let's look at specific anomalies to understand what the model is catching.

In [ ]:
print("=== Voltage Drops (Below 190V) ===")
print("These are clearly wrong - normal charging is 208-240V")
print()
display(df_feat[df_feat['voltage_out_of_range'] == 1][['station_id','timestamp','voltage','current','power_kw','temperature_c','message']].head(5))

print("\n=== Overheating (Above 75°C) ===")
print("The charger is getting too hot")
print()
display(df_feat[df_feat['overtemp'] == 1][['station_id','timestamp','voltage','current','power_kw','temperature_c','message']].head(5))

print("\n=== Error Codes ===")
print("The charger reported a fault")
print()
display(df_feat[df_feat['has_error'] == 1][['station_id','timestamp','voltage','current','power_kw','error_code','message']].head(5))